In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# label_path = 'path to lables'
label_path = 'labels_path'
class_names = [
    'Band_Neutrophil', 'Giant_Platelet', 'Monocyte',
    'Normal_Lymphocyte', 'Normal Platelet', 'Reactive_Lymphocyte',
    'Segmented_Neutrophil'
]

data = []

if not os.path.exists(label_path):
    print(f"Directory '{label_path}' not found. Please update the label_path variable.")
else:
    for file in os.listdir(label_path):
        if file.endswith('.txt'):
            with open(os.path.join(label_path, file), 'r') as f:
                for line in f.readlines():
                    parts = line.split()
                    if len(parts) == 5:
                        cid = int(parts[0])
                        data.append([
                            class_names[cid] if cid < len(class_names) else f"ID {cid}",
                            float(parts[1]), float(parts[2]),
                            float(parts[3]), float(parts[4])
                        ])

if not data:
    print("No data found. Ensure the path is correct and files contain YOLO formatted text.")
else:
    df = pd.DataFrame(data, columns=['Class', 'x', 'y', 'w', 'h'])


    plt.figure(figsize=(6, 6))
    plt.scatter(df['w'], df['h'], s=15, alpha=0.6, c='#022A6E', edgecolors='#ffffff', linewidths=0.8)
    plt.xlabel('Normalized Width', fontsize=12, fontweight='bold')
    plt.ylabel('Normalized Height', fontsize=12, fontweight='bold')
    plt.xlim(0, 0.25)
    plt.ylim(0, 0.5)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('width_height_distribution.png', dpi=300)
    plt.show()
    print("Saved: width_height_distribution.png")



    class_order = [
        'Platelet', 'Normal_Lymphocyte', 'Band_Neutrophil',
        'Reactive_Lymphocyte', 'Segmented_Neutrophil', 'Giant_Platelet', 'Monocyte'
    ]
    df['Class'] = pd.Categorical(df['Class'], categories=class_order, ordered=True)
    df = df.sort_values('Class')

    marker_sizes = {
        'Platelet': 13,
        'Normal_Lymphocyte': 50,
        'Band_Neutrophil': 50,
        'Reactive_Lymphocyte': 50,
        'Segmented_Neutrophil': 55,
        'Giant_Platelet': 55,
        'Monocyte': 60
    }


    fig, ax = plt.subplots(figsize=(10, 8.5))


    sns.set_theme(style="white")


    sns.scatterplot(
        data=df,
        x='x',
        y='y',
        hue='Class',
        style='Class',
        size='Class',
        sizes=marker_sizes,
        alpha=0.9,
        palette='Dark2',
        edgecolor='#ffffff',
        linewidth=1.2,
        ax=ax
    )

    plt.xlabel('X Center (Normalized)', fontsize=13, fontweight='bold', labelpad=12)
    plt.ylabel('Y Center (Normalized)', fontsize=13, fontweight='bold', labelpad=12)

    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')
        spine.set_linewidth(1.5)

    plt.xlim(0, 1.0)
    plt.ylim(0, 1.0)
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)


    plt.legend(
        title="Class",
        title_fontsize='12',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        frameon=True,
        facecolor='#f8f9fa',
        edgecolor='#cccccc',
        markerscale=1.8
    )


    plt.tight_layout()
    plt.savefig('center_point_distribution_premium.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: center_point_distribution_premium.png")

Tebular Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

file_name = '/content/Tebular_Dataset.csv'
df = pd.read_csv(file_name)

print("=== 1. Dataset Overview ===")
print(f"Shape of the dataset: {df.shape[0]} rows and {df.shape[1]} columns\n")
print("First 5 rows of the dataset:")
print(df.head(5))
print("\n" + "="*50 + "\n")

print("=== 2. Data Types & Missing Values ===")
print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\n" + "="*50 + "\n")

print("=== 3. Summary Statistics ===")
df1=df.drop(columns=['Gender','Dengue'])
print(df1.describe().T)
print("\n" + "="*50 + "\n")



### Z-Score Normalization (Standardization)


In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# Create a copy of the dataframe so we don't alter the original

# 1. Apply Binary Encoding for Gender and Dengue
# Mapping Male=1, Female=0 and Positive=1, Negative=0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Dengue'] = df['Dengue'].map({'Positive': 1, 'Negative': 0})

# 2. Identify numerical columns to normalize
# We explicitly exclude the newly encoded 'Gender' and 'Dengue' columns
all_numeric = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
exclude_cols = ['Gender', 'Dengue']
columns_to_normalize = [col for col in all_numeric if col not in exclude_cols]

# 3. Initialize and apply the StandardScaler
scaler = StandardScaler()

# Apply normalization only to the filtered list
if columns_to_normalize: # Check added to prevent errors if no columns exist
    df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

print(f"Z-score normalization complete for: {columns_to_normalize}")
print(f"Binary encoding applied to: {exclude_cols}")

display(df.head())

# Verify statistics
if columns_to_normalize:
    print("\nSummary of Normalized Columns (Mean should be ~0, Std should be 1):")
    display(df[columns_to_normalize].describe().T[['mean', 'std']])

print("\nSummary of Encoded Columns (Should be 0s and 1s):")
display(df[exclude_cols].describe().T[['mean', 'min', 'max']])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


X = df.drop(columns=['Dengue'])
y = df['Dengue']

feature_names = X.columns.tolist()

# Normalize data using MinMaxScaler (Chi-Square requires non-negative attributes)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)

# Initialize a dataframe to keep track of scores from all methods
ranking_df = pd.DataFrame(index=feature_names)

# 5 FEATURE SELECTION METHODS

# Method 1: Chi-Square (X²) Test
# Computes the chi2 statistic between each feature and the target
chi2_selector = SelectKBest(score_func=chi2, k='all')
chi2_selector.fit(X_scaled, y)
ranking_df['Chi_Square'] = chi2_selector.scores_

# Method 2: Pearson's Correlation Coefficient
# Computes the absolute correlation of each feature with the target variable
correlations = [abs(df[col].corr(y)) for col in feature_names]
ranking_df['Pearson_Corr'] = correlations

# Method 3: Recursive Feature Elimination (RFE)
# Prunes features step-by-step using an estimator (Logistic Regression)
rfe_estimator = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=1)
rfe.fit(X_scaled, y)
# Invert ranking so that the most important features have higher numbers
ranking_df['RFE_Score'] = len(feature_names) - rfe.ranking_ + 1

# Method 4: Logistic Regression (Weights/Coefficients)
# Extracts the absolute weight magnitude for each normalized feature
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_scaled, y)
ranking_df['Log_Reg_Weights'] = np.abs(lr_model.coef_[0])

# Method 5: Random Forest (Feature Importance)
# Calculates average impurity reduction across an ensemble of decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_scaled, y)
ranking_df['Random_Forest'] = rf_model.feature_importances_


# CONSENSUS THRESHOLDING

# Normalize all raw metric scores between 0 and 1 so they can be fairly averaged
normalized_ranking = (ranking_df - ranking_df.min()) / (ranking_df.max() - ranking_df.min())

# Calculate the ensemble average score for each feature across all 5 strategies
ranking_df['Average_Ensemble_Score'] = normalized_ranking.mean(axis=1)

# Sort features by highest average consensus
ranking_df = ranking_df.sort_values(by='Average_Ensemble_Score', ascending=False)

# Define the baseline global threshold (the mean score of all features)
global_threshold = ranking_df['Average_Ensemble_Score'].mean()

# Filter features that outperform the consensus criteria
selected_features = ranking_df[ranking_df['Average_Ensemble_Score'] >= global_threshold]


# DISPLAY RESULTS & CUSTOMIZED GRAPH

print("--- ALL FEATURE RANKINGS ---")
print(ranking_df[['Average_Ensemble_Score']])

print(f"\n--- THRESHOLD CRITERIA ---")
print(f"Global Consensus Threshold (Mean): {global_threshold:.4f}")

sns.set_theme(style="whitegrid")
plt.figure(figsize=(11, 6), dpi=100)

colors = ['#1f77b4' if x >= global_threshold else '#b0bec5' for x in ranking_df['Average_Ensemble_Score']]

ax = sns.barplot(
    x=ranking_df['Average_Ensemble_Score'],
    y=ranking_df.index,
    palette=colors,
    hue=ranking_df.index,
    legend=False
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=6, fontsize=9, fontweight='semibold', color='#37474f')


plt.axvline(
    x=global_threshold,
    color='#d32f2f',
    linestyle='--',
    linewidth=1.8,
    label=f'Consensus Threshold ({global_threshold:.3f})'
)

plt.xlabel('Normalized Consensus Score (0 - 1)', fontsize=11, fontweight='semibold', labelpad=10, color='#37474f')
plt.ylabel('Features', fontsize=11, fontweight='semibold', labelpad=10, color='#37474f')

plt.xticks(fontsize=10)
plt.yticks(fontsize=10, fontweight='medium')

plt.xlim(0, max(ranking_df['Average_Ensemble_Score']) * 1.12)

plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#cfd8dc', fontsize=10)

sns.despine(left=True, bottom=True)

plt.tight_layout()


plt.savefig('feature_importance_profile.png', dpi=1000, bbox_inches='tight', transparent=False)


plt.show()